# 22 — Caveat Resolution and Quality Recode v1

This notebook turns the blunt `clean/caveated` split into a more useful confidence framework.

Purpose:

- separate **technical caveats** from genuinely serious interpretive caveats;
- reclassify county-derived evidence as **medium confidence** rather than automatically “major caveat”;
- keep serious boundary/data problems visible;
- output revised review lists and map-ready files for the report.

This notebook does not change the underlying model score. It changes how uncertainty is labelled and presented.

In [2]:
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
DICTIONARY_DIR = DATA_DIR / "dictionaries"

for d in [PROCESSED_DIR, GEOGRAPHY_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed


In [3]:
OUTPUT_DIR = PROCESSED_DIR / "target_review_pack_v1_revised_caveats"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NORTH_WEST_REVIEW_FILENAME = "north_west_consolidated_target_review_v1.csv"
ALL_REVIEW_FILENAME = "all_available_consolidated_target_review_v1.csv"

print("Output:", OUTPUT_DIR)

Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats


In [4]:
def find_file(filename, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = [
            PROCESSED_DIR / "target_review_pack_v1",
            PROCESSED_DIR / "target_model_v2",
            PROCESSED_DIR / "target_model_v1",
            PROCESSED_DIR / "report_assets_v1",
            PROCESSED_DIR / "election_results",
            PROCESSED_DIR,
            DATA_DIR / "raw" / "election_results",
            DATA_DIR / "raw",
            PROJECT_DIR,
            Path.cwd(),
        ]
    for folder in search_dirs:
        path = folder / filename
        if path.exists():
            return path
    # recursive fallback under processed and data/raw
    for root in [PROCESSED_DIR, DATA_DIR / "raw"]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(f"Could not find {filename}")
    return None


def read_csv(filename, required=True):
    path = find_file(filename, required=required)
    if path is None:
        print("Optional file missing:", filename)
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def save_csv(df, path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)
    print("Saved:", path, df.shape)
    return path


def boolish(s):
    return s.fillna(False).astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def safe_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

## 22.1 Load consolidated review tables

Run Notebook 20 first. This notebook prefers `north_west_consolidated_target_review_v1.csv`, with optional all-available processing if the national consolidated file exists.

In [5]:
nw, nw_path = read_csv(NORTH_WEST_REVIEW_FILENAME)
all_review, all_path = read_csv(ALL_REVIEW_FILENAME, required=False)

print("North West rows:", len(nw))
if all_review is not None:
    print("All-available rows:", len(all_review))

Loaded north_west_consolidated_target_review_v1.csv: (825, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_consolidated_target_review_v1.csv
Loaded all_available_consolidated_target_review_v1.csv: (7572, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\all_available_consolidated_target_review_v1.csv
North West rows: 825
All-available rows: 7572


## 22.2 Caveat recode rules

The key change is this:

- **county-derived electoral evidence** becomes `Medium confidence`, not a serious caveat by default;
- **Sefton / explicit boundary uncertainty** remains visible;
- missing or broken election data remains serious;
- general model-limit caveats stay in method notes rather than row-level warnings.

In [6]:
def recode_caveats(df):
    df = df.copy()

    # Standardise booleans / strings.
    for col in ["has_major_caveat", "target_model_ready", "is_clean_watchlist", "is_caveated_watchlist", "is_breakthrough_complacency", "is_demographic_build"]:
        if col in df.columns:
            df[col] = boolish(df[col])

    boundary_text = safe_col(df, "boundary_caveat", "").fillna("").astype(str)
    county_text = safe_col(df, "county_election_caveat", "").fillna("").astype(str)
    confidence_note = safe_col(df, "data_confidence_note", "").fillna("").astype(str)

    latest_year = to_num(safe_col(df, "latest_election_source_year", np.nan))
    valid_votes = to_num(safe_col(df, "latest_election_allocated_valid_votes", np.nan))
    margin = to_num(safe_col(df, "latest_election_margin_pct_allocated", np.nan))

    # Caveat classes.
    df["technical_caveat_flag"] = False
    df["boundary_caveat_flag"] = boundary_text.str.strip().ne("")
    df["county_election_evidence_flag"] = county_text.str.strip().ne("") | ((latest_year == 2025) & df.get("LAD25NM", pd.Series("", index=df.index)).isin([
        "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde", "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley", "Rossendale", "South Ribble", "West Lancashire", "Wyre"
    ]))
    df["missing_or_invalid_election_flag"] = valid_votes.fillna(0).le(0) | margin.isna()
    df["sefton_boundary_flag"] = df.get("LAD25NM", pd.Series("", index=df.index)).eq("Sefton")

    # Evidence type.
    df["electoral_evidence_type"] = np.select(
        [
            df["missing_or_invalid_election_flag"],
            df["county_election_evidence_flag"],
            latest_year.notna(),
        ],
        ["missing_or_invalid", "county_derived", "ward_level_or_bestfit"],
        default="unknown",
    )

    # Report confidence.
    serious = df["missing_or_invalid_election_flag"] | (df["boundary_caveat_flag"] & df["sefton_boundary_flag"])
    medium = (~serious) & (df["county_election_evidence_flag"] | df["boundary_caveat_flag"])

    df["report_confidence_band"] = np.select(
        [serious, medium],
        ["Serious caveat / manual review", "Medium confidence"],
        default="High confidence",
    )

    df["report_caveat_level"] = np.select(
        [serious, medium],
        ["serious", "note"],
        default="none",
    )

    def caveat_summary(row):
        notes = []
        if row.get("missing_or_invalid_election_flag", False):
            notes.append("missing/invalid election metrics")
        if row.get("sefton_boundary_flag", False):
            notes.append("Sefton boundary change risk")
        elif row.get("boundary_caveat_flag", False):
            notes.append("boundary caveat")
        if row.get("county_election_evidence_flag", False):
            notes.append("county-derived electoral evidence")
        if not notes:
            return "No material row-level caveat"
        return "; ".join(notes)

    df["report_caveat_summary"] = df.apply(caveat_summary, axis=1)

    # Revised lane: turn many former caveated rows into medium-confidence opportunity, not a separate scary category.
    def revised_lane(row):
        lane = str(row.get("strategic_lane", "Monitor"))
        if row["report_confidence_band"] == "Serious caveat / manual review":
            return "Manual Review / Serious Caveat"
        if lane == "Caveated Opportunity" and row["report_confidence_band"] == "Medium confidence":
            return "Medium-Confidence Opportunity"
        return lane

    df["revised_strategic_lane"] = df.apply(revised_lane, axis=1)

    # Report inclusion flag.
    df["include_in_main_report"] = df["report_confidence_band"].isin(["High confidence", "Medium confidence"])
    df["include_in_serious_caveat_appendix"] = df["report_confidence_band"].eq("Serious caveat / manual review")

    return df

nw_revised = recode_caveats(nw)
summary = nw_revised.groupby(["report_confidence_band", "revised_strategic_lane"], dropna=False).size().reset_index(name="rows")
display(summary.sort_values(["report_confidence_band", "rows"], ascending=[True, False]))

if all_review is not None:
    all_revised = recode_caveats(all_review)
else:
    all_revised = None

,report_confidence_band,revised_strategic_lane,rows
0,Serious caveat / manual review,Manual Review / Serious Caveat,825


## 22.3 Save revised outputs

In [7]:
save_csv(nw_revised, OUTPUT_DIR / "north_west_revised_consolidated_review_v1.csv")

save_csv(
    nw_revised[nw_revised["report_confidence_band"].eq("High confidence")],
    OUTPUT_DIR / "north_west_high_confidence_review_v1.csv"
)

save_csv(
    nw_revised[nw_revised["report_confidence_band"].eq("Medium confidence")],
    OUTPUT_DIR / "north_west_medium_confidence_review_v1.csv"
)

save_csv(
    nw_revised[nw_revised["report_confidence_band"].eq("Serious caveat / manual review")],
    OUTPUT_DIR / "north_west_serious_caveat_manual_review_v1.csv"
)

# Opportunity-specific outputs.
save_csv(
    nw_revised[nw_revised["revised_strategic_lane"].isin(["Clean Opportunity", "Medium-Confidence Opportunity"])].sort_values("initial_watchlist_score", ascending=False),
    OUTPUT_DIR / "north_west_revised_opportunity_watchlist_v1.csv"
)

save_csv(
    nw_revised[nw_revised["revised_strategic_lane"].eq("Breakthrough Build")].sort_values("breakthrough_complacency_score", ascending=False),
    OUTPUT_DIR / "north_west_revised_breakthrough_build_v1.csv"
)

save_csv(
    nw_revised[nw_revised["revised_strategic_lane"].eq("Long-Term Demographic Build")].sort_values("demographic_relevance_score", ascending=False),
    OUTPUT_DIR / "north_west_revised_demographic_build_v1.csv"
)

# Map-ready slim file.
map_cols = [c for c in [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region",
    "strategic_lane", "revised_strategic_lane", "report_confidence_band", "report_caveat_level", "report_caveat_summary",
    "initial_watchlist_score", "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "breakthrough_complacency_score",
    "dominant_cluster_name", "latest_election_top_party_bucket", "electoral_evidence_type"
] if c in nw_revised.columns]
save_csv(nw_revised[map_cols], OUTPUT_DIR / "north_west_revised_confidence_map_ready_v1.csv")

summary = nw_revised.groupby(["report_confidence_band", "revised_strategic_lane"], dropna=False).size().reset_index(name="rows")
save_csv(summary, OUTPUT_DIR / "north_west_caveat_recode_summary_v1.csv")

if all_revised is not None:
    save_csv(all_revised, OUTPUT_DIR / "all_available_revised_consolidated_review_v1.csv")
    all_summary = all_revised.groupby(["analysis_region", "report_confidence_band", "revised_strategic_lane"], dropna=False).size().reset_index(name="rows")
    save_csv(all_summary, OUTPUT_DIR / "all_available_caveat_recode_summary_v1.csv")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_revised_consolidated_review_v1.csv (825, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_high_confidence_review_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_medium_confidence_review_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_serious_caveat_manual_review_v1.csv (825, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_revised_opportunity_watchlist_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1_revised_caveats\north_west_revised_breakthrough_build_v1.csv (0, 59)
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\ta

## 22.4 Interpretation note

Use `High confidence` and `Medium confidence` rows in the main report. Keep `Serious caveat / manual review` rows in an appendix or internal QA note unless the issue is strategically important.